# LASSI-Verify: Emperical Verification

**Argo Gateway API** Jupyter Notebook version

+ ***Post LASSI-Judge version with data collection for pass@k and energy_reduction@k***
+ No translation
+ Source code scripts from:
https://github.com/zjin-lcf/HeCBench/tree/master/src

NVIDIA GPU power measurements: https://pypi.org/project/nvidia-ml-py/  (pynvml)

Added support for AMD GPU device measurements and HIP code (ROCm)

In [1]:
from lassi.gpu_detector import detect_gpu_vendors

# Detect available GPUs
vendors = detect_gpu_vendors()
if not vendors:
    print("❌ No GPUs detected! LASSI requires GPU hardware.")
    raise SystemExit("Pipeline stopped - no GPU hardware found")

detected_vendor = vendors[0]['vendor']
gpu_count = vendors[0]['count']
print(f"🔍 Detected: {detected_vendor.upper()} with {gpu_count} GPU(s)")

# Vendor-specific initialization
if detected_vendor == "nvidia":
    # NVIDIA-specific setup
    import pynvml
    pynvml.nvmlInit()
    print("NVIDIA Driver Version:", pynvml.nvmlSystemGetDriverVersion())
    pynvml.nvmlShutdown()
    
    # Show NVIDIA system info
    !nvidia-smi
    !nvcc --version
    
elif detected_vendor == "amd":
    # AMD-specific setup
    from lassi.amd_power import amd_report_gpu_memory
    
    # Show AMD system info
    !rocminfo | head -20
    !rocm-smi
    !hipcc --version
    
print(f"\n✅ {detected_vendor.upper()} GPU environment ready for LASSI!")

🔍 Detected: NVIDIA with 1 GPU(s)
NVIDIA Driver Version: 570.211.01
Fri Mar 20 15:47:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 5000 Ada Gene...    Off |   00000000:01:00.0 Off |                  Off |
| N/A   38C    P8              3W /   80W |      14MiB /  16376MiB |      0%      Default |
|                                         |                        |     

## Setup

In [2]:
import os
import json
import csv
import time
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

# Import LASSI modules
import lassi.llm_config
# import lassi.pipeline # The Pipeline script calls the other LASSI libraries not imported here.
import lassi.pipeline_v4
from lassi.format_colors import color
from lassi.prompt_dictionary import PromptDictionary
from lassi.monitor import start_monitor, log_figure

# Enable the autoreload extension
%load_ext autoreload
%reload_ext autoreload

# Reload all modules imported with %aimport every time before executing subsequent code.
%autoreload 1

# We need to reload the libraries while under development
%aimport lassi.format_colors
%aimport lassi.prompt_dictionary
%aimport lassi.llm_config
%aimport lassi.llm_methods
%aimport lassi.pipeline_v4
%aimport lassi.get_power
%aimport lassi.argostream_curl
%aimport lassi.code_manager
%aimport lassi.pipeline_methods
%aimport lassi.verify
%aimport lassi.monitor

In [3]:
# ── LASSI Monitor — live output dashboard ──────────────────────────────────────
# Starts a local web server and streams all pipeline print() output to a browser
# tab in real-time.  Run this cell once after the imports cell above.
# Then open the link below in a new browser tab and keep it visible.
#
# If matplotlib figures are produced (EE task), they appear automatically in a
# sidebar panel on the right side of the dashboard — no extra calls needed.
#
# The dashboard accumulates the full session history, so a late-opened tab
# will still see everything printed since start_monitor() was called.
#
# SSH remote access: forward port 5007 to your local machine, then open
#   http://localhost:5007  in your local browser.
start_monitor()   # default port 5007 — pass port=XXXX to change

'http://localhost:5007'

## Code Generation Experiment - Setup + Run

_____
***STEP 1/2: Input experimental configuration parameters in the following method.***

In [ ]:
def experimental_setup():
    # Load config file with source application names and input parameters
    with open("source_apps.json", "r") as f:
        source_apps = json.load(f)
    
    """
    [1] ENTER APPLICATION FOLDER NAME FROM HeCBench
        >> Execution input parameters provided.
    
    *** USE FOR LASSI-Verfify experiment: ***
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
     (this one may show a lot of failures)

    HeCBench apps:
    -- matrix-rotate ["30000", "100"]  ["10000", "1"]  # Usage: %s <matrix size> <repeat>
    -- jacobi [""] (none required)
    -- layout ["2500"] ["1"]
    -- atomicCost ["16", "10"] ["1"]
    -- segment-reduce ["16384", "100"] (a replacement for atomicCost, same "Data compression and reduction" category in HeCBench)
       --> (may be trivial for LASSI-Verify)
    -- dense-embedding ["100000", "128", "100"] ["10000", "8", "1"]
    -- pathfinder ["100000", "10000", "2500"] ["100000", "1000", "5"] ["10000", "1000", "1000"]  # Usage: <column length> <row length> <pyramid_height>
    -- bsearch ["8388608", "1000"] ["10000", "1"]  # Usage: <number of elements> <repeat>
    -- entropy ["10000", "1024", "1"] ["8192", "8192", "500"]   # Usage: <width> <height> <repeat>
        --> can't get a passing source_verify
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
    -- randomAccess ["10"] ["1"]  # Usage: <repeat>
    -- lid-driven-cavity [""]
    -- murmurhash3 ["750000, 500"]
    -- chacha20 ["300000"]
    -- jaccard ["1024, 512, 1000"]
    -- keogh ["256", "22500000", "100"]
    -- floydwarshall ["2048", "100", "16"]
    -- extrema ["750"]
    -- threadfence ["100", "250000000"]
    -- all-pairs-distance ["10000"]
    -- marchingCubes ["100"]

    Altis apps:
    -- level1/bfs ["--size 3"]
    -- level2/raytracing [""]
    -- level2/mandelbrot ["--imageSize 1024 --iterations 100000"]

    Miniapps:
    -- miniMDock ["-lfile", "./input/7cpa/7cpa_ligand.pdbqt", "-nrun 500"]
    -- XSBench ["-m", "event", "-l", "1700000000"]
    
    """

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # Enter source application to process through LASSI
    source_collection = "HeCBench"  # "Miniapps" "HeCBench"
    source_app = "entropy"  # "miniMDock" jacobi
    make_script = False  # Used only for Miniapps
    build_script = False  # False (for HeCBench) Use for Altis only
    

    """
    Valid source code folder names (under source_codes):
    1. HeCBench
    2. altis/src/cuda
    3. miniMDock/bin/
    """
    source_folder = source_apps[source_collection]["source_folder"]  # "altis/src/cuda/level2"    #  "HeCBench" "altis/src/cuda/level1"
    app_folder_name = source_apps[source_collection][source_app]["app_folder"]
    lines_of_code = source_apps[source_collection][source_app]["code_lines"]
    if make_script:
        app_name = source_app  # app_folder_name
        file_name = "Main"  # kernels
    else:
        app_name = "mt_" + app_folder_name + "-cuda_main"
        file_name = ""

    
    """
    [2] SELECT THE CODE TRANSLATION MAPPING DIRECTION EXPERIMENT CONFIGURATION
    
    CUDA -- OMP -- HIP -- SYCL -- FORTRAN
    """   
    # These should be the same if *not* doing translation.
    source_lang = "CUDA"
    target_lang = "CUDA"

    verifyprompt = True  # True or False -- Print out LLM prompt in stdout to preview
    
    # TOGGLE USE CODE KNOWLEDGE CONTEXT FOR TARGET
    use_context_summaries_only = True  # True or False
    few_shot_learning = False  # True or False
    
    # LLM judge code analysis (pre-cursor to LASSI-Verify)
    run_code_analysis = False  # True or False
    
    # THESE WILL NO LONGER BE OPTIONS
    # test_code_first = True  # True or False
    use_context = False  # True or False -- NOT USED IN v2, v3
    use_self_prompting = False  # True or False -- NOT USED IN v2, v3
    baseline_oneshot_goal = True  # True or False -- NOT USED IN v2, v3
    
    # -----------------------------------------------------------------------
    # TASK MODE CONFIGURATION
    # Set these two flags together depending on the task:
    #
    #   ENERGY EFFICIENCY (EE) task:
    #     power_measure    = True   -- measure GPU power/energy during execution
    #     yes_power_feedback = True -- run LASSI self-improvement loop using energy as the optimization metric
    #
    #   RUNTIME PERFORMANCE task:
    #     power_measure    = False  -- skip power measurement, use wall-clock duration instead
    #     yes_power_feedback = True -- run LASSI self-improvement loop using runtime as the optimization metric
    #
    #   CODE TRANSLATION task (one-shot, no iterative improvement):
    #     power_measure    = False  -- no measurement needed
    #     yes_power_feedback = False -- skip feedback loop entirely (generate once, compile, execute, judge, done)
    #     NOTE: translation mode still has a known infinite-loop-on-success issue when yes_power_feedback=False
    # -----------------------------------------------------------------------
    power_measure = False  # True (EE task) | False (runtime or translation task)
    vary_llm_temp = True  # True or False

    # TOGGLE LASSI SELF-IMPROVEMENT FEEDBACK LOOP
    yes_power_feedback = True  # True (EE or runtime task) | False (translation task, one-shot only)

    # -----------------------------------------------------------------------
    # LASSI-VERIFY TIER 2 CONFIGURATION
    # run_tier2=True: insert passthrough observation blocks and validate empirical equivalence
    # tier2_tolerance: relative floating-point tolerance for passthrough comparison (default 1e-6)
    # -----------------------------------------------------------------------
    run_tier2 = True         # True to enable Tier 2 empirical equivalence checking
    tier2_tolerance = 1e-6    # Relative tolerance for passthrough block comparison

    # LASSI-VERIFY TIER 3 CONFIGURATION
    # run_tier3=True: generate metamorphic contract and run variant filter (requires run_tier2=True)
    # tier3_tolerance: relative tolerance for metamorphic relation checks (default 1e-5, slightly looser than Tier 2)
    # -----------------------------------------------------------------------
    run_tier3 = True        # True to enable Tier 3 metamorphic testing
    tier3_tolerance = 1e-5   # Relative tolerance for metamorphic relation comparison

    # -----------------------------------------------------------------------
    # LLM-AS-A-JUDGE CONFIGURATION
    # run_llm_judge=True: enable LLM functional equivalence judge (pre-Tier-1/2 stop-gap)
    # run_llm_judge=False: pure three-tier mode — Tier 1 + Tier 2 determine validity only
    # -----------------------------------------------------------------------
    run_llm_judge = True     # True to enable LLM judge; False for pure three-tier LASSI-Verify mode

    # ASSIGN CODE COMPILE COMMAND
    code_build_script = ""
    code_compiler = ""
    code_compiler_kwds = ""
    if source_lang == "CUDA":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=CUDA DEVICE=GPU CARD=NVIDIA NUMWI=64"
            elif source_app == "XSBench":
                code_compiler = "cuda/make"
        else:
            code_compiler = "nvcc"
            code_compiler_kwds = "-std=c++14 -Xcompiler -Wall -arch=sm_89 -O3"
            # -arch=sm_70 for aiops01 GPU (Argonne)  # -arch=sm_80 for A100s
            # -arch=sm_60 for P100
            # -arch=sm_89 for RTX 5000
        if yes_power_feedback:
            context_knowledge_selection = "cuda_optimization_summary"
            context_fewshot_examples = "cuda_c_few_shot_examples"
        else:
            context_knowledge_selection = "cuda"
    elif source_lang == "OMP":
        # For A100 on Linux_x86_64 or Linux_aarch64
        # code_compiler = "g++" # "/opt/nvidia/hpc_sdk/Linux_aarch64/2024/compilers/bin/nvc++"  # 
        # code_compiler_kwds = "-fopenmp -O3" # "-Wall -O3 -Minfo -mp=gpu -gpu=cc80"  # -c
        # !! TODO - REVIEW: code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc80"  # "-fopenmp -O3 -c"
        # For RTX 5000 Ada on Linux x86_64:
        code_compiler = "/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++" # "g++"
        code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc89"  # "-fopenmp -O3"
        context_knowledge_selection = "omp"
    elif source_lang == "HIP":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=HIP DEVICE=GPU CARD=AMD NUMWI=64 HIP_PATH=/opt/rocm-6.3.3"
            elif source_app == "XSBench":
                code_compiler = "hip/make"
            else:
                code_compiler_kwds = ""
        else:
            code_compiler = "hipcc"
            code_compiler_kwds = "-std=c++14 -Wall -O3 --amdgpu-target=gfx908"
        if yes_power_feedback:
            context_knowledge_selection = "hip_optimization_summary"
            context_fewshot_examples = "hip_c_few_shot_examples"
        else:
            context_knowledge_selection = "hip"  # TODO

    # TOGGLE EXECUTE PIPELINE
    yes_execute = True  # True / False
    if yes_execute:
        # Enter execution input parameters, if needed
        # A list of parameter string values, e.g., ["10000", "1
        # ****************************
        # ---> EXPERIMENT UPDATE FIELD - Now, automated:
        exe_input_parameters = source_apps[source_collection][source_app]["app_params"]
    else:
        exe_input_parameters = [""]

    
    """
    [3] SELECT THE LLM TO USE FOR THE CODE TRANSLATION EXPERIMENT
    """
    # SELECT A MODEL

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # -- FROM ARGO:
    # model = "gpt35"  # (max token = 4096)
    # model = "gpt35large"  # (max token = 16384)
    # model = "gpt4large"  # (max token = 32768)
    # model = "gpt4turbo"  # (max token = 128000)
    # model = "gpto4mini"  # (max token = 128000)
    # model_coder = "gpto4mini"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4o"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 200000)
    # model_coder = "gpt4olatest"  # (max token = 200000)
    model = "gpt54"
    model_coder = "gpt54"
    model_judge = "gpt41"  # (max token = 200000)

    # -- FROM CEREBRAS
    # model = "gpt-oss-120b"
    # model_judge = "gpt-oss-120b"
    # model_coder = "qwen-3-coder-480b"
    
    # -- FROM OLLAMA
    # model = "devstral:24b" # (max token = 128000)
    # model_coder = "devstral:24b" # (max token = 128000)
    # model_judge = "devstral:24b" # (max token = 128000)
    # model = "qwq:latest"  # (max token = 128000)
    # model_coder = "qwen2.5-coder:32b"  # (max token = 32768)
    # model_judge = "qwq:latest"  # (max token = 128000)
    # model = "qwq:latest"  # (max token = 131072)
    # model = "gemma3:27b"  # crashing on code gen
    # model_coder = "gemma3:27b"
    # model = "granite3.2:8b-instruct-q8_0"
    # model_coder = "granite-code:34b-instruct-q8_0"
    # model = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model = "dolphin3"  # (max token = 131072)
    # model = "llama3.3:latest"  # (max token = 131072)
    # model = "phi4" # (max token = 16384)
    # model_coder = "codestral:22b"  # (max token = 32768)
    # model = "codestral:22b-v0.1-q8_0"  # (max token = 32768)
    # model = "wizardcoder:33b-v1.1-q8_0"  # (max token = 16384)
    # model_coder = "deepseek-coder:33b-instruct-q8_0"  # (max token = 16384)
    # model = "deepseek-coder-v2:16b-lite-instruct-fp16"  # (max token = 128000)
    # model = "deepseek-coder-v2:16b-lite-instruct-q8_0"
    # model = "phi3:14b"  # (max token = 131072)
    # model_judge = "llama3.3:70b"  # (max token = 128000)
    # model = "qwen3:latest"
    # model_coder = "qwen3:latest"
    # model_judge = "qwen3:latest"  # (max token = 40960)
    # model_coder = "qwen3-coder:30b"  # context 262144
    # model = "gpt-oss:20b"
    # model_judge = "gpt-oss:20b"
  
    # Set maximum context token count size for selected LLM:
    num_ctx = 200000
    num_ctx_coder = 200000

    # Needed only for Argo endpoints
    uri = "local"  # local  remote
    lassi.llm_config.argo_username = "mdearing"  # Argo Gateway username

    # ASSIGN LLM API SOURCE:
    if model in ["gpt54", "gpt5", "gpt35", "gpt35large", "gpt4", "gpt4large", "gpt4turbo", "gpt4o", "gpt4olatest", "gpt41",
                 "gpto1preview", "gpto1mini", "gpto3mini", "gpto4mini", "claudesonnet4", "claudesonnet46"]:
        llm_source = "argo"
    elif model in ["gpt-oss-120b", "qwen-3-coder-480b"]:
        llm_source = "cerebras"
    elif ("GGUF" in model and "hf.co" not in model):
        llm_source = "lmstudio"
    else:
        llm_source = "ollama"

    """
    [4] SET ADDITIONAL PIPELINE PARAMETERS
    """
    experiment = "TEST_FULLEXP_verify_0318-2026_" + model_coder + "_" + app_folder_name + "_runtm_" + detected_vendor.upper()  # Assign any label for experiment name.
    display_response = True
    test_with_error = False
    generate_test_case = False

    # Unload any previously loaded LLMs in the Ollama server 
    # (clear out the memory)
    if llm_source == "ollama":
        lassi.llm_config.unload_ollama_model(model)
        lassi.llm_config.unload_ollama_model(model_coder)

    """
    [5] SET DATA COLLECTION CONFIGURATION
    """
    # Data spreadsheet column headers
    DATAFIELDNAMES = [
        "trial_number",
        "timestamp",
        "experiment",
        "gpu_vendor",
        "app_name",
        "code_line_count",
        "llm",
        "llm_coder",
        "llm_judge",
        "source_v0_energy_J",
        "source_v0_avg_power_W",
        "source_v0_runtime_s",
        "source_verify_runtime_s",  # ★ instrumented source runtime
        "base_v1_energy_J",
        "base_v1_avg_power_W",
        "base_v1_runtime_s",
        "base_v1_valid",
        "base_v1_energy_diff",
        "base_v1_avg_power_diff",
        "base_v1_runtime_diff",
        "zeroshot_runtime_s",        # ★ schema-aligned alias for base_v1_runtime_s
        "zeroshot_compile_status",   # ★ "success" | "compile_error" | "skipped"
        "zeroshot_execute_status",   # ★ "success" | "execute_error" | "skipped"
        # "base_v1_optimization_methods",
        # "base_v2_energy_J",
        # "base_v2_avg_power_W",
        # "base_v2_runtime_s",
        # "base_v2_valid",
        # "base_v2_selfcorrections",
        # "base_v2_energy_diff",
        # "base_v2_avg_power_diff",
        # "base_v2_runtime_diff",
        # "base_v2_optimization_methods",
        "lassiee_energy_J",
        "lassiee_avg_power_W",
        "lassiee_runtime_s",
        "lassiee_valid",
        "lassiee_version",
        "lassiee_llm_temp",
        "lassiee_selfcorrections",
        "lassiee_eeiterations",
        "lassiee_energy_diff",
        "lassiee_avg_power_diff",
        "lassiee_runtime_diff",
        "lassiee_optimization_methods",
        "total_selfcorrections",
        "compile_selfcorrections",   # ★
        "execute_selfcorrections",   # ★
        "tier1_selfcorrections",     # ★
        "tier2_selfcorrections",     # ★
        "tier3_selfcorrections",     # ★
        "global_counter_at_end",     # ★ global verify counter at trial end
        "total_code_versions",       # ★ distinct LASSI code block indices
        "total_code_attempts",       # ★ rows in attempt log for this trial
        "total_eeiterations",
        "pipeline_runtime_min",
        "tier2_pass_count",
        "tier2_fail_total",
        "tier3_pass_count",
        "tier3_fail_total",
        "tier3_active_relations",
        "tier3_inactive_relations",  # ★
        "judge_false_positives",
        "judge_false_negatives",
        "lassiee_clean_status",
        "lassiee_clean_runtime_s",
        "lassiee_clean_runtime_diff",
        "lassiee_clean_energy_J",
        "lassiee_clean_energy_diff",
        "base_llm_tier1_verdict",
        "base_llm_tier2_verdict",
        "base_llm_tier3_verdict",
        "base_llm_verified_valid"
    ]
    
    return (app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder, llm_source, use_context, use_context_summaries_only,
            few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler, code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters,
            experiment, display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal, use_self_prompting, vary_llm_temp,
            verifyprompt, DATAFIELDNAMES, detected_vendor.upper(),
            run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
            run_llm_judge)

# EXPERIMENTAL DATA COLLETION MANAGEMENT
def init_data_collection(dir_path, DATAFIELDNAMES, filename="result_lassi-verify_pipeline_data.csv"):
    os.makedirs(dir_path, exist_ok=True)
    fp = os.path.join(dir_path, filename)
    if not os.path.exists(fp):
        with open(fp, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=DATAFIELDNAMES)
            writer.writeheader()
    return fp

_____
***STEP 2/2: Run the following block to implement the pipeline with the configuration parameters.***

In [ ]:
# Experimental repeat for pass@k metrics
trial_number = 1
total_trials = 3

# Load prompt dictionary (to capture recent manual changes to script, if any)
prompts = PromptDictionary()

all_trials_start = time.time()

# Store the current home working directory (needed for code execution)
original_home_directory = os.getcwd()


# COLLECT THE EXPERIMENT CONFIGURATIONS
(
    app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder,
    llm_source, use_context, use_context_summaries_only, few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler,
    code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters, experiment,
    display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal,
    use_self_prompting, vary_llm_temp, verifyprompt, DATAFIELDNAMES, gpu_vendor,
    run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
    run_llm_judge
) = experimental_setup()

# IMPORT CONTEXT KNOWLEDGE FOR TARGET PROGRAMMING LANGUAGE
if gpu_vendor == "NVIDIA":
    input_context = "[BACKGROUND KNOWLEDGE: CUDA C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("cuda") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: CUDA C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]
elif gpu_vendor == "AMD":
    input_context = "[BACKGROUND KNOWLEDGE: HIP C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("hip") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: HIP C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]


# DEFINE PROMPT TEMPLATES
# TODO: MOST TASK PROMPT TEMPLATES WILL BE HANDLED BY THE PIPELINE METHODS
# If we are doing the runtime task, then set the system prompts, otherwise they are for ee
# we need to bring back in the translation systems prompt reference when we create LASSI-X
if (power_measure == False and yes_power_feedback == True):
    system_prompt_selection_index = "general_system_" + source_lang + "_runtime"
else:
    system_prompt_selection_index = source_lang + "_ee"  # "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
this_llm_system_prompt = prompts.get_system_prompt(str(system_prompt_selection_index))

coderefactor_prompt_selection_index = source_lang  # + "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
if (not baseline_oneshot_goal and use_self_prompting):
    # TODO: THIS SHOULD BE A DEAD BRANCH BECAUSE WE DON'T WANT TO USE baseline_oneshot_goal ANYMORE
    his_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index))
else: 
    if (power_measure == False and yes_power_feedback == True):
        # RUNTIME TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_runtime"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext_runtime"))
    else:
        # EE TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext"))

# LOAD INPUT CODE TO REFACTOR
input_code_folder = source_folder + "/" + app_folder_name + "/"
if build_script:
    input_code_filename = app_name
elif make_script:
    input_code_filename = file_name
else:
    input_code_filename = app_folder_name + "-" + source_lang.lower() + "_main"
if source_lang == "OMP" or source_lang == "SYCL":
    input_code_extension = "cpp"
elif source_lang == "CUDA" or source_lang == "HIP":
    if (make_script and source_lang == "HIP"):
        input_code_extension = "cpp"  # custom for miniMDock and XSBench
    else:
        input_code_extension = "cu"
if make_script:
    source_file_directory = "generated_codes/" + input_code_folder
else:
    source_file_directory = "source_codes/" + input_code_folder
if make_script:
    # SELECT FILE TO REFACTOR from miniapp
    if source_lang == "CUDA":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/cuda/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    elif source_lang == "HIP":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/hip/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_file_subdirectory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
else:
    input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
with open(source_file_path, 'r') as file:
    input_code = file.read()
file.close()


# SPECIFY OUTPUT CODE SAVE
if build_script:
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
    output_code_file_name = app_name
elif make_script:
    output_code_file_name = file_name
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
else:
    output_code_directory = "generated_codes/" + llm_source + "_" + model.replace('/', '--') + "/promptset_" + str(coderefactor_prompt_selection_index)
    output_code_file_name = input_code_filename + "_ee_refactorcode_" + source_lang + "_exp" + experiment

now = datetime.now(ZoneInfo("America/Chicago"))
dt_string = now.strftime("%m-%d-%Y_t%H_%M_%S")

# SPECIFIY DATA COLLECTION SAVE
experiment_data_directory = "generated_codes/experiment/" + experiment + "__" + dt_string
# Convert to absolute directory::
experiment_data_directory = os.path.abspath(experiment_data_directory)
experiment_data_file_object = init_data_collection(experiment_data_directory, DATAFIELDNAMES)
experiment_data = dict.fromkeys(DATAFIELDNAMES, None) # all keys start out as None
# Table 2 ★ per-attempt log — one row per code generation attempt (including self-correction retries)
attempt_log_file_path = init_data_collection(
    experiment_data_directory, lassi.pipeline_v4.ATTEMPT_LOG_FIELDNAMES,
    filename="result_lassi-verify_attempt_log.csv"
)
experiment_data["gpu_vendor"] = str(detected_vendor.upper())

while trial_number <= total_trials:
    # For the miniApp, we need to reset the source kernel file
    #  because the pipeline for this scenario must overwrite the 
    #  source code in the generated_codes folder.
    if make_script:
        if source_lang == "CUDA":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels_orig_merged.cu"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main_orig_merged.cu"
        elif source_lang == "HIP":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels_orig_merged.cpp"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main_orig_merged.cpp"
        try:
            print(f"[reset] Copying clean kernel -> working file\n"
                  f"        src: {clean_kernel_file}\n"
                  f"        dst: {working_kernel_file}")
            shutil.copyfile(clean_kernel_file, working_kernel_file)
            print("[reset] Success: working file reset to clean copy.")
        except FileNotFoundError as e:
            print(f"[reset] ERROR: file not found: {e.filename}")
        except PermissionError as e:
            print(f"[reset] ERROR: permission denied: {e.filename}")
        except OSError as e:
            # catches other IO errors (disk full, cross-device issues, etc.)
            print(f"[reset] ERROR: {e.__class__.__name__}: {e}")

    # CREATE METADATA RECORD CONTENT
    meta_data_file = "***** EXPERIMENT CONFIGURATION SUMMARY *****\n"
    meta_data_file += "\nGPU Vendor: " + str(detected_vendor.upper())
    meta_data_file += "\nDate-time: " + dt_string
    meta_data_file += "\nApplication: " + app_folder_name
    meta_data_file += "\n\nRefactoring task: " + system_prompt_selection_index
    meta_data_file += "\n\nLLM Source: " + llm_source
    meta_data_file += "\nModel: " + model
    meta_data_file += "\nModel coder: " + model_coder
    meta_data_file += "\nModel judge: " + model_judge
    meta_data_file += "\nSystem Prompt Index: " + str(coderefactor_prompt_selection_index)
    meta_data_file += "\nCode file template: " + output_code_directory + "/" + output_code_file_name
    meta_data_file += "\n\nLLM System Prompt: " + "\n\n" + this_llm_system_prompt
    # meta_data_file += "\n\nLLM Prompt for Code Generation: " + "\n\n" + this_llm_refactoredcode_prompt
    meta_data_file += "\n\nPower measurement: " + str(power_measure)
    
    # Display metadata
    print("\n" + meta_data_file + "\n")
    
    print("\n\n" + color.BLUE + color.BOLD + "***** BEGIN THE LASSI ENERGY EFFICIENCY PIPELINE (v4) *****" + color.END + " Trial #: " + str(trial_number) + "\n\n")
    
    # RUN LASSI-EE PIPELINE
    complete = lassi.pipeline_v4.auto_code_llmgeneration_pipeline(
        prompts,
        this_llm_system_prompt,
        this_llm_refactoredcode_prompt,
        llm_source, 
        model,
        model_coder,
        model_judge,
        num_ctx,
        num_ctx_coder,
        # arrayormatrix_size_range,
        # thread_count_range,
        use_context,
        use_context_summaries_only,
        few_shot_learning,
        run_code_analysis,
        input_context,
        input_code_file_subdirectory,
        input_code,
        lines_of_code,
        # target_code,
        target_lang,
        input_code_extension,
        source_lang,
        code_compiler,
        code_compiler_kwds,
        build_script,
        make_script,
        code_build_script,
        output_code_directory,
        output_code_file_name,
        source_file_path,
        display_response,
        test_with_error,
        generate_test_case,
        yes_execute,
        exe_input_parameters,
        meta_data_file,
        power_measure,
        yes_power_feedback,
        dt_string,
        baseline_oneshot_goal,
        use_self_prompting,
        vary_llm_temp,
        experiment,
        verifyprompt,
        trial_number,
        experiment_data_directory,
        experiment_data_file_object,
        experiment_data,
        app_name,
        run_tier2=run_tier2,
        tier2_tolerance=tier2_tolerance,
        run_tier3=run_tier3,
        tier3_tolerance=tier3_tolerance,
        run_llm_judge=run_llm_judge,
        attempt_log_file_path=attempt_log_file_path
    )
    
    # Reset to original working directory
    os.chdir(original_home_directory)
    
    if complete:
        print("\n\n" + color.GREEN + color.BOLD + "***** LASSI PIPELINE COMPLETED *****\n\n" + color.END)
    else:
        print("\n\n" + color.RED + color.BOLD + "***** LASSI PIPELINE FAILED TO COMPLETE *****\n\n" + color.END)

    # Increment experiment counter
    trial_number = trial_number + 1


all_trials_end = time.time() - all_trials_start

print("\n\n" + color.PURPLE + color.BOLD + "***** ALL TRIALS COMPLETED ***** Total time (min): " + str(round(all_trials_end/60, 2)) + "\n\n" + color.END)




***** EXPERIMENT CONFIGURATION SUMMARY *****

GPU Vendor: NVIDIA
Date-time: 03-20-2026_t15_47_17
Application: entropy

Refactoring task: general_system_CUDA_runtime

LLM Source: argo
Model: gpt54
Model coder: gpt54
Model judge: gpt41
System Prompt Index: CUDA
Code file template: generated_codes/argo_gpt54/promptset_CUDA/entropy-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_entropy_runtm_NVIDIA

LLM System Prompt: 

You are an expert C++/CUDA engineer. Your goal is to refactor provided CUDA C++ code to reduce wall-clock execution time by improving GPU and CPU utilization and parallelism, while preserving all original functionality and expected output of the provided code. You must also carefully read and incorporate the best practices and CUDA optimization techniques from the provided background documents and refactoring plans. If the source code contains LASSI-VERIFY instrumentation blocks (marked with ASSERTIONS, PASSTHROUGH, or METAMORPHIC headers), copy ever

Waiting for the system to settle down for 1 seconds...: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/entropy/entropy-cuda_main
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/entropy
executable:  entropy-cuda_main
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/entropy
Execute command: ['./entropy-cuda_main', '10000', '1024', '1']

running: executable -  entropy-cuda_main

----- < BEGIN: STANDARD OUTPUT FROM CODE EXECUTION > -----

Average kernel (baseline) execution time 0.003416 (s)
Average kernel (optimized) execution time 0.000667 (s)
PASS

THE RETURN CODE:  0

----- < END: Standard Output > -----


Program executed successfully!

----- END OF STAGE 0: TEST SOURCE -----

None

*** DEBUG: code_version: 0 | tracking_best_version: 0 | code_blocks len: 1 (next idx: 1) ***

--> ... prompt LLM to describe source code (Tier 1/2/3 context).


=====> VERIFY THE PROMPTING <===== [toke

This program is a CUDA benchmark and correctness test for computing a local entropy value at every element of a 2D image-like grid whose values are integers in the range `[0, 15]`.

## High-level functionality

It:

1. Reads `width`, `height`, and `repeat` from the command line.
2. Creates a random 2D input array of `char` values in `[0, 15]`.
3. Copies that input to the GPU.
4. Runs two CUDA kernels:
   - a baseline entropy kernel,
   - an optimized entropy kernel.
5. Measures and prints the average execution time of each kernel over `repeat` launches.
6. Copies the optimized kernel’s output back to the CPU.
7. Verifies the GPU result against a CPU reference implementation from `reference.h`.
8. Prints `PASS` or `FAIL`.

---

## What “entropy” means here

For each pixel/location `(x, y)`, the program looks at a `5x5` neighborhood centered on that location, clipped at image boundaries. It counts how many times each of the 16 possible symbol values appears in that neighborhood. From those counts, it computes the Shannon entropy:

\[
H = -\sum_k p_k \log_2(p_k)
\]

where `p_k = count[k] / total`, and `total` is the number of valid neighbors in the local window.

So the output is a `float` entropy map with one entropy value per input element.

---

## Functional blocks

## 1. Headers and dependencies

The code includes:

- `stdio.h`, `stdlib.h` for I/O and memory allocation,
- `math.h` for `log2f` and `fabsf`,
- `chrono` for timing,
- `cuda.h` for CUDA runtime interaction,
- `reference.h` for the CPU reference implementation used in verification.

### External dependency
- `reference(output_ref, input, height, width);`

This function is not defined in the shown file, but it is clearly intended to compute the correct CPU result for comparison.

---

## 2. Baseline CUDA kernel: `entropy`

### Signature
```cpp
__global__ 
void entropy(float *d_entropy, const char *d_val, int height, int width)
```

### Purpose
Computes the local entropy for each output pixel independently using straightforward per-thread logic.

### Inputs
- `d_entropy`: device output array of `float`
- `d_val`: device input array of `char`
- `height`, `width`: image dimensions

### Per-thread behavior
Each thread corresponds to one image coordinate:
- `x = threadIdx.x + blockIdx.x * blockDim.x`
- `y = threadIdx.y + blockIdx.y * blockDim.y`

### Core algorithm
For its assigned pixel, the thread:
1. Creates a local histogram:
   - `char count[16]`
   - initialized to zero
2. Scans a `5x5` window around `(x, y)`:
   - `dy = -2..2`
   - `dx = -2..2`
3. For each valid neighbor inside bounds:
   - increments the histogram bin corresponding to `d_val[yy * width + xx]`
   - increments `total`
4. Computes entropy:
   - if `total < 1`, it forces `total = 1`
   - otherwise, for each of 16 bins:
     - computes `p = count[k] / total`
     - accumulates `-p * log2f(p)`
5. Writes the entropy to `d_entropy[y * width + x]` if `(x, y)` is in bounds.

### Notes on implementation
- The neighborhood is clipped at borders.
- The input values are assumed to be in `[0, 15]`; this is required because they index directly into `count[16]`.
- The code relies on floating-point behavior for `p = 0`, where `p * log2f(p)` is mathematically treated as zero; in practice, this can be delicate numerically, but the reference presumably matches this behavior closely enough for the chosen threshold.
- The histogram and total are stored in `char`, which is safe here because the maximum count is 25 for a `5x5` window.

---

## 3. Optimized CUDA kernel: `entropy_opt`

### Signature
```cpp
template<int bsize_x, int bsize_y>
__global__ void entropy_opt(
    float *d_entropy,
    const char *d_val,
    const float *d_logTable,
    int m, int n)
```

### Purpose
Computes the same entropy values as the baseline kernel, but uses a reformulated entropy expression and shared-memory-backed counting storage to reduce some computation cost.

### Template parameters
- `bsize_x`, `bsize_y`: block dimensions, used for compile-time sizing of shared memory.

In this program it is instantiated as:
```cpp
entropy_opt<16, 16>
```

### Inputs
- `d_entropy`: device output array
- `d_val`: device input array
- `d_logTable`: precomputed lookup table for `i * log2(i)` for `i = 0..25`
- `m`, `n`: height and width

### Shared memory
```cpp
__shared__ int sd_count[16][bsize_y*bsize_x];
```

This allocates histogram storage for all threads in the block:
- 16 bins,
- one per thread index within the block.

So each thread has a private 16-bin histogram stored in shared memory rather than local stack memory.

### Per-thread indexing
- `x`, `y` are global coordinates
- `idx = threadIdx.y * bsize_x + threadIdx.x` is the thread’s linear index inside the block

### Core algorithm
Each thread:
1. Zeros its own 16 histogram bins in shared memory:
   ```cpp
   sd_count[i][idx] = 0
   ```
2. Scans the same `5x5` neighborhood around `(x, y)`.
3. For each valid neighbor:
   - increments `sd_count[d_val[yy*n+xx]][idx]`
   - increments `total`
4. Computes entropy using a transformed formula:
   ```cpp
   entropy -= d_logTable[sd_count[k][idx]];
   entropy = entropy / total + log2f(total);
   ```

### Mathematical basis
Because:
\[
H = -\sum_k \frac{c_k}{T}\log_2\left(\frac{c_k}{T}\right)
\]
this can be rearranged to:
\[
H = \log_2(T) - \frac{1}{T}\sum_k c_k \log_2(c_k)
\]
where terms with `c_k = 0` or `1` contribute zero.

The lookup table stores:
- `0` for `i = 0, 1`
- `i * log2(i)` for `i >= 2`

This avoids repeated per-bin probability division and repeated `log2f` calls for counts.

### Output
Writes the result to `d_entropy[y*n+x]` if in bounds.

### Notes
- Functionally equivalent to the baseline kernel for this problem.
- Uses shared memory, but not for inter-thread reuse of image data; rather, it stores per-thread histograms in shared memory.
- The lookup table size is 26 because the max neighborhood size is 25.

---

## 4. `main` function

### Signature
```cpp
int main(int argc, char* argv[])
```

### Purpose
Drives setup, execution, timing, result retrieval, and verification.

---

## Detailed behavior of `main`

### A. Argument parsing
It expects exactly 3 arguments:
- `width`
- `height`
- `repeat`

If not provided, it prints:
```cpp
Usage: %s <width> <height> <repeat>
```
and exits with status 1.

---

### B. Host memory allocation
It computes:
- `input_bytes = width * height * sizeof(char)`
- `output_bytes = width * height * sizeof(float)`

Allocates:
- `input`: host input image
- `output`: host buffer for GPU output
- `output_ref`: host buffer for reference output

---

### C. Input initialization
It seeds the random number generator with:
```cpp
srand(123);
```

Then fills every input element with:
```cpp
rand() % 16
```

So the input is deterministic across runs.

---

### D. Device memory allocation and input transfer
Allocates:
- `d_input` on device
- `d_output` on device

Copies host input to device:
```cpp
cudaMemcpy(d_input, input, input_bytes, cudaMemcpyHostToDevice);
```

---

### E. Launch configuration
Uses:
- block size: `16 x 16`
- grid size:
  - `((width + 15) / 16, (height + 15) / 16)`

This covers the full image with possible excess threads at edges, guarded by bounds checks.

---

### F. Baseline kernel timing
It synchronizes, starts a host timer, launches the baseline kernel `repeat` times, synchronizes again, and computes elapsed time.

Printed output:
```cpp
Average kernel (baseline) execution time %f (s)
```

Important detail:
- The timing measures total launch+execution time over repeated kernel invocations, then divides by `repeat`.
- It uses `std::chrono::steady_clock` on the host, not CUDA events.

---

### G. Optimized kernel preparation
Creates a host lookup table:
```cpp
float logTable[26];
for (int i = 0; i <= 25; i++) logTable[i] = i <= 1 ? 0 : i*log2f(i);
```

This table supports the entropy reformulation in the optimized kernel.

Allocates `d_logTable` on the device and copies the table to it.

---

### H. Optimized kernel timing
Similarly to the baseline:
- synchronizes,
- times `repeat` launches of:
  ```cpp
  entropy_opt<16, 16>
  ```
- synchronizes,
- prints the average time:
```cpp
Average kernel (optimized) execution time %f (s)
```

---

### I. Copy output back to host
After the optimized kernel finishes, the program copies `d_output` into `output`.

So the verification compares the optimized GPU result against the CPU reference.

---

### J. Device memory cleanup
Frees:
- `d_input`
- `d_output`
- `d_logTable`

---

### K. Correctness verification
Calls:
```cpp
reference(output_ref, input, height, width);
```

Then compares every element of `output` against `output_ref`.

Comparison criterion:
```cpp
fabsf(output[...] - output_ref[...]) > 1e-3f
```

If any mismatch exceeds the tolerance:
- `ok = false`

Finally prints:
- `"PASS"` if all values match within tolerance
- `"FAIL"` otherwise

---

### L. Host memory cleanup and exit
Frees:
- `input`
- `output`
- `output_ref`

Returns `0`.

---

## Major functions

## `entropy`
Baseline CUDA kernel that computes local entropy directly using per-thread local histograms and explicit probability/log calculations.

## `entropy_opt<bsize_x, bsize_y>`
Templated CUDA kernel that computes the same entropy using:
- per-thread histograms stored in shared memory,
- a precomputed lookup table for `count * log2(count)`,
- the transformed entropy formula.

## `main`
Program entry point handling:
- argument parsing,
- random input generation,
- GPU memory management,
- kernel timing,
- result retrieval,
- CPU reference validation,
- pass/fail reporting.

## `reference`
External CPU reference function included from `reference.h`; used to validate GPU correctness.

---

## Major data structures

## Host arrays
- `char* input`
  - Input image/grid values, each in `[0, 15]`
- `float* output`
  - Output copied back from the GPU
- `float* output_ref`
  - CPU reference output

## Device arrays
- `char* d_input`
  - GPU copy of input data
- `float* d_output`
  - GPU output entropy map
- `float* d_logTable`
  - GPU lookup table used by the optimized kernel

## Per-thread histogram structures
### In baseline kernel
- `char count[16]`
  - Local per-thread histogram of symbol counts in the `5x5` window

### In optimized kernel
- `__shared__ int sd_count[16][bsize_y*bsize_x]`
  - Shared memory storage containing one 16-bin histogram per thread in the block

## Lookup table
- `float logTable[26]`
  - Host-side table where:
    - `logTable[0] = 0`
    - `logTable[1] = 0`
    - `logTable[i] = i * log2(i)` for `i >= 2`

Used to avoid recomputing these values inside the optimized kernel.

---

## Computational pattern

The computation is a stencil-like local neighborhood operation:
- each output depends on a local `5x5` region of input,
- border handling is done by bounds checks,
- neighboring threads read heavily overlapping regions of the input.

This overlap is not explicitly exploited by shared input tiling in the current implementation; instead, the “optimization” focuses on simplifying the entropy arithmetic and moving histogram storage into shared memory.

---

## Output behavior

The program prints three lines of interest:

1. Average baseline kernel time
2. Average optimized kernel time
3. `PASS` or `FAIL`

Example structure:
- `Average kernel (baseline) execution time ...`
- `Average kernel (optimized) execution time ...`
- `PASS`

---

## Assumptions and constraints implicit in the code

- Input values must be in `[0, 15]`, because they directly index 16-bin histograms.
- The local neighborhood is fixed at `5x5`.
- Maximum valid count is 25, which is why the lookup table has size 26.
- `char` is sufficient for counts and totals because values never exceed 25.
- The optimized kernel’s formula assumes `total > 0`; in practice every in-bounds pixel has at least itself in the window, so for valid output positions this holds.
- Threads outside image bounds still execute the counting loops, but the boundary checks prevent invalid reads, and the output write is guarded.

---

## Overall design summary

This is a small self-contained CUDA benchmarking program comparing two GPU implementations of local entropy over a 2D discrete-valued image:

- the first is a direct, straightforward implementation,
- the second uses algebraic reformulation plus a lookup table and shared-memory-backed per-thread counting.

The program is structured as:
1. setup and initialization,
2. baseline benchmark,
3. optimized benchmark,
4. correctness verification against CPU reference,
5. pass/fail reporting.

Its primary purpose is both performance comparison and correctness validation of the optimized GPU entropy computation.


----- STARTING CODE INSERT - Invariant Assertions -----


=====> VERIFY THE PROMPTING <===== [tokens: 19,915]

TEMPERATURE: 0.2

SYSTEM: You are an expert C++/CUDA engineer performing surgical code insertion. Your ONLY task is to INSERT new instrumentation code at specified locations in an existing source file. You MUST preserve ALL existing code EXACTLY as written — do not modify, simplify, reorder, optimize, or remove any existing code, comments, blank lines, or whitespace. Every single line of the input file must appear in your output, in the same order. You always surround your generated code with the three characters ```.

PROMPT: 
    You are a GPU code verification specialist.

    Your task is to instrument the provided CUDA C++ source code with **transformation-invariant
    assertion checks** — properties that must hold in the original code AND any correct refactored
    version, regardless of parallelization strategy, memory layout, or algorithmic optimization.

    These a

```cpp
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <chrono>
#include <cuda.h>
#include "/home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/entropy/reference.h"

__global__ 
void entropy(
        float *__restrict__ d_entropy,
    const char*__restrict__ d_val, 
    int height, int width)
{
  const int x = threadIdx.x + blockIdx.x * blockDim.x;
  const int y = threadIdx.y + blockIdx.y * blockDim.y;

  // value of matrix element ranges from 0 inclusive to 16 exclusive
  char count[16];
  for (int i = 0; i < 16; i++) count[i] = 0;

  // total number of valid elements
  char total = 0;

  // 5x5 window
  for(int dy = -2; dy <= 2; dy++) {
    for(int dx = -2; dx <= 2; dx++) {
      int xx = x + dx;
      int yy = y + dy;
      if(xx >= 0 && yy >= 0 && yy < height && xx < width) {
        count[d_val[yy * width + xx]]++;
        total++;
      }
    }
  }

  float entropy = 0;
  if (total < 1) {
    total = 1;
  } else {
    for(int k = 0; k < 16; k++) {
      float p = __fdividef((float)count[k], (float)total);
      entropy -= p * log2f(p);
    }
  }

  if(y < height && x < width) d_entropy[y * width + x] = entropy;
}

template<int bsize_x, int bsize_y>
__global__ void entropy_opt(
       float *__restrict__ d_entropy,
  const  char*__restrict__ d_val, 
  const float*__restrict__ d_logTable,
  int m, int n)
{
  __shared__ int sd_count[16][bsize_y*bsize_x];

  const int x = threadIdx.x + blockIdx.x * blockDim.x;
  const int y = threadIdx.y + blockIdx.y * blockDim.y;
  const int idx = threadIdx.y*bsize_x + threadIdx.x;

  for(int i = 0; i < 16;i++) sd_count[i][idx] = 0;

  char total = 0;
  for(int dy = -2; dy <= 2; dy++) {
    for(int dx = -2; dx <= 2; dx++) {
      int xx = x + dx,
          yy = y + dy;

      if(xx >= 0 && yy >= 0 && yy < m && xx < n) {
        sd_count[d_val[yy*n+xx]][idx]++;
        total++;
      }
    }
  }

  float entropy = 0;
  for(int k = 0; k < 16; k++)
    entropy -= d_logTable[sd_count[k][idx]];
  
  entropy = entropy / total + log2f(total);
  if(y < m && x < n) d_entropy[y*n+x] = entropy;
}

int main(int argc, char* argv[]) {
  if (argc != 4) {
    printf("Usage: %s <width> <height> <repeat>\n", argv[0]);
    return 1;
  }
  const int width = atoi(argv[1]); 
  const int height = atoi(argv[2]); 
  const int repeat = atoi(argv[3]); 

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate dimensions and repeat count at function entry
  {
    bool _t1_pass = (width > 0 && height > 0 && repeat > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_dims_repeat_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  const int input_bytes = width * height * sizeof(char);
  const int output_bytes = width * height * sizeof(float);
  char* input = (char*) malloc (input_bytes);
  float* output = (float*) malloc (output_bytes);
  float* output_ref = (float*) malloc (output_bytes);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate host allocations and byte sizes
  {
    bool _t1_pass = (input_bytes > 0 && output_bytes > 0 && input != NULL && output != NULL && output_ref != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_alloc_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  srand(123);
  for (int i = 0; i < height; i++)
    for (int j = 0; j < width; j++)
      input[i * width + j] = rand() % 16;

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate input value range before host-to-device transfer
  {
    bool _t1_pass = true;
    for (int i = 0; i < height * width; i++) {
      int _t1_v = (int)input[i];
      if (!(_t1_v >= 0 && _t1_v < 16)) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_input_range_pre_htod]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  char* d_input;
  cudaMalloc((void**)&d_input, input_bytes);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate input buffer before host-to-device transfer
  {
    bool _t1_pass = (d_input != NULL && input != NULL);
    if (_t1_pass) {
      for (int i = 0; i < height * width; i++) {
        int _t1_v = (int)input[i];
        if (!(_t1_v >= 0 && _t1_v < 16)) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_htod_input_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaMemcpy(d_input, input, input_bytes, cudaMemcpyHostToDevice);

  float* d_output;
  cudaMalloc((void**)&d_output, output_bytes);

  dim3 grids ((width+15)/16, (height+15)/16);
  dim3 blocks (16, 16);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate device output allocation and kernel launch configuration
  {
    bool _t1_pass = (d_output != NULL && blocks.x > 0 && blocks.y > 0 && grids.x > 0 && grids.y > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_kernel_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // baseline kernel
  cudaDeviceSynchronize();
  auto start = std::chrono::steady_clock::now();
  
  for (int i = 0; i < repeat; i++)
    entropy <<< grids, blocks >>> (d_output, d_input, height, width);

  cudaDeviceSynchronize();
  auto end = std::chrono::steady_clock::now();
  auto time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  printf("Average kernel (baseline) execution time %f (s)\n", (time * 1e-9f) / repeat);

  // optimized kernel

  float logTable[26];
  for (int i = 0; i <= 25; i++) logTable[i] = i <= 1 ? 0 : i*log2f(i);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate lookup table values before host-to-device transfer
  {
    bool _t1_pass = true;
    for (int i = 0; i <= 25; i++) {
      if (!isfinite(logTable[i])) { _t1_pass = false; break; }
      if (i <= 1) {
        if (!(fabsf(logTable[i]) <= 1e-6f)) { _t1_pass = false; break; }
      } else {
        if (!(logTable[i] >= 0.0f)) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_logtable_pre_htod_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  float* d_logTable;
  cudaMalloc((void**)&d_logTable, sizeof(logTable));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate lookup table device pointer before host-to-device transfer
  {
    bool _t1_pass = (d_logTable != NULL);
    if (_t1_pass) {
      for (int i = 0; i <= 25; i++) {
        if (!isfinite(logTable[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_logtable_buffer_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaMemcpy(d_logTable, logTable, sizeof(logTable), cudaMemcpyHostToDevice);
 
  cudaDeviceSynchronize();
  start = std::chrono::steady_clock::now();

  for (int i = 0; i < repeat; i++)
    entropy_opt<16, 16> <<< grids, blocks >>> (d_output, d_input, d_logTable, height, width);

  cudaDeviceSynchronize();
  end = std::chrono::steady_clock::now();
  time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  printf("Average kernel (optimized) execution time %f (s)\n", (time * 1e-9f) / repeat);

  cudaMemcpy(output, d_output, output_bytes, cudaMemcpyDeviceToHost);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate output entropy values after device-to-host transfer
  {
    bool _t1_pass = true;
    for (int i = 0; i < height * width; i++) {
      if (!isfinite(output[i])) { _t1_pass = false; break; }
      if (!(output[i] >= 0.0f && output[i] <= logTable[25] + 1e-3f)) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_output_post_dtoh_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaFree(d_input);
  cudaFree(d_output);
  cudaFree(d_logTable);

  // verify
  reference(output_ref, input, height, width);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate reference output numerical properties
  {
    bool _t1_pass = true;
    for (int i = 0; i < height * width; i++) {
      if (!isfinite(output_ref[i])) { _t1_pass = false; break; }
      if (!(output_ref[i] >= 0.0f && output_ref[i] <= logTable[25] + 1e-3f)) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_reference_output_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  bool ok = true;
  for (int i = 0; i < height; i++) {
    for (int j = 0; j < width; j++) {
      if (fabsf(output[i * width + j] - output_ref[i * width + j]) > 1e-3f) {
        ok = false; 
        break;
      }
    }
    if (!ok) break;
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate final comparison result at function exit
  {
    bool _t1_pass = ok;
    printf("LASSI-VERIFY ASSERTIONS [main_final_result_match]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  printf("%s\n", ok ? "PASS" : "FAIL");
 
  free(input);
  free(output);
  free(output_ref);
  return 0;
}
```

... (17 outputs left out)

This program is a CUDA benchmark and correctness test for computing a per-pixel local entropy map over a 2D input image-like array whose values are discrete symbols in the range `[0, 15]`.

## Overall functionality

Given:
- `width`
- `height`
- `repeat`

the program:

1. Allocates a `height x width` input array of `char`.
2. Fills it with random values from `0` to `15`.
3. Copies that input to the GPU.
4. Runs two CUDA kernels repeatedly:
   - a **baseline entropy kernel**
   - an **optimized entropy kernel**
5. Measures and prints average execution time for each kernel.
6. Copies the final GPU result back to the CPU.
7. Computes a reference result on the CPU using `reference(...)`.
8. Compares the GPU output against the CPU reference.
9. Prints `PASS` if all values match within tolerance, otherwise `FAIL`.

The entropy being computed at each output element is the Shannon entropy of the values in a local **5x5 neighborhood** centered at that element, with boundary handling by ignoring out-of-range neighbors.

---

## Functional blocks

## 1. Headers and dependencies

The program includes:
- Standard C/C++ headers:
  - `stdio.h`, `stdlib.h`, `math.h`
  - `chrono` for timing
- CUDA header:
  - `cuda.h`
- External reference implementation:
  - `reference.h`

### Purpose
These provide:
- memory allocation and I/O
- math operations like `log2f`, `fabsf`
- host-side timing
- CUDA runtime functionality
- a trusted CPU implementation for verification

---

## 2. Baseline CUDA kernel: `entropy`

### Signature
- `__global__ void entropy(float* d_entropy, const char* d_val, int height, int width)`

### Purpose
Computes the local entropy for each `(x, y)` position in the input matrix using a straightforward implementation.

### How it works
Each GPU thread corresponds to one output element:
- `x = threadIdx.x + blockIdx.x * blockDim.x`
- `y = threadIdx.y + blockIdx.y * blockDim.y`

For that position, it:
1. Creates a local histogram `count[16]` for the 16 possible input values.
2. Scans the surrounding **5x5 window** (`dx, dy` from `-2` to `2`).
3. For each valid neighbor within image bounds:
   - increments the appropriate histogram bin
   - increments `total`, the count of valid neighbors
4. Computes Shannon entropy:
   - for each symbol `k`:
     - `p = count[k] / total`
     - `entropy -= p * log2f(p)`
5. Writes the result to `d_entropy[y * width + x]` if the thread is inside bounds.

### Important details
- Boundary pixels use smaller effective windows because out-of-range neighbors are skipped.
- `count` and `total` are stored as `char`, which is sufficient because the maximum 5x5 window size is 25.
- If `total < 1`, it forces `total = 1`, though in normal valid-image use a thread corresponding to a real pixel should always have at least itself in range.
- It does not explicitly guard against `p == 0` before evaluating `log2f(p)`, so mathematically it relies on behavior that may produce the correct result in context but is numerically less careful than the optimized version.

---

## 3. Optimized CUDA kernel: `entropy_opt`

### Signature
- `template<int bsize_x, int bsize_y>`
- `__global__ void entropy_opt(float* d_entropy, const char* d_val, const float* d_logTable, int m, int n)`

### Purpose
Computes the same local entropy values as the baseline kernel, but uses a more efficient formulation intended to reduce computational cost.

### Key optimization idea
Instead of directly computing:
- `-sum(p * log2(p))`

it uses the identity:
- `H = log2(total) - (1 / total) * sum(count_k * log2(count_k))`

The values `count * log2(count)` for possible counts `0..25` are precomputed in a lookup table `d_logTable`.

### How it works
For each thread:
1. Uses shared memory:
   - `__shared__ int sd_count[16][bsize_y*bsize_x];`
   - Each thread has one slot per symbol bin, indexed by its linear local thread id.
2. Initializes its per-thread 16 histogram counters in shared memory to zero.
3. Scans the same 5x5 neighborhood.
4. For each valid neighbor:
   - increments the corresponding histogram count in shared memory
   - increments `total`
5. Computes:
   - `entropy -= d_logTable[sd_count[k][idx]]` for all 16 bins
   - then transforms:
     - `entropy = entropy / total + log2f(total)`
6. Writes result if `(x, y)` is in bounds.

### Parameters
- `m` and `n` correspond to image dimensions:
  - `m = height`
  - `n = width`

### Why it is considered optimized
Compared with the baseline:
- avoids repeated `log2f` calls per bin-probability term
- avoids floating-point division per histogram bin
- uses a precomputed table for `count * log2(count)`
- reformulates entropy computation to be cheaper

### Shared memory use
Although called optimized, shared memory here is used as per-thread histogram storage rather than for inter-thread reuse of neighborhood pixels. So the optimization is mostly arithmetic rather than window-data sharing.

---

## 4. Main program: `main`

### Signature
- `int main(int argc, char* argv[])`

### Purpose
Drives the entire benchmark, input generation, kernel execution, timing, and correctness checking.

---

## 5. Command-line parsing

The program expects exactly 3 arguments:
- `<width>`
- `<height>`
- `<repeat>`

If the count is wrong, it prints:
- `Usage: %s <width> <height> <repeat>`

and exits with status `1`.

### Parsed values
- `width = atoi(argv[1])`
- `height = atoi(argv[2])`
- `repeat = atoi(argv[3])`

---

## 6. Host memory allocation

The program allocates:
- `input` as a `char*` of size `width * height`
- `output` as a `float*` of size `width * height`
- `output_ref` as a `float*` of size `width * height`

### Purpose
- `input`: host-side random test data
- `output`: result copied back from GPU
- `output_ref`: CPU reference result

---

## 7. Input initialization

The random seed is fixed:
- `srand(123)`

Then every input element is assigned:
- `rand() % 16`

### Purpose
This creates reproducible test input where every value is one of the 16 histogram classes.

---

## 8. Device memory allocation and host-to-device transfer

Allocated on the GPU:
- `d_input` for the input image
- `d_output` for the entropy output

Then:
- `cudaMemcpy(d_input, input, input_bytes, cudaMemcpyHostToDevice)`

### Purpose
Makes the input available to CUDA kernels and reserves space for results.

---

## 9. CUDA launch configuration

The kernel launch configuration is:
- `blocks(16, 16)`
- `grids((width+15)/16, (height+15)/16)`

### Purpose
Creates a 2D grid of 16x16 thread blocks to cover the whole image, including partial edge blocks.

---

## 10. Baseline kernel benchmark section

### Flow
1. `cudaDeviceSynchronize()` before timing.
2. Records start time with `std::chrono::steady_clock`.
3. Repeats:
   - `entropy<<<grids, blocks>>>(...)`
4. Synchronizes again after all launches.
5. Measures elapsed time.
6. Prints average time per launch.

### Printed output
- `Average kernel (baseline) execution time ... (s)`

### Purpose
Measures wall-clock execution time of the baseline GPU implementation.

---

## 11. Log lookup table construction for optimized kernel

A host array:
- `float logTable[26];`

is filled for counts `0..25`:
- `0` for `i <= 1`
- otherwise `i * log2f(i)`

Then it is copied to GPU memory `d_logTable`.

### Why size 26
A 5x5 window can contain at most 25 valid elements, so histogram counts range from 0 to 25 inclusive.

### Purpose
Supports the optimized entropy formula without recomputing logarithms for histogram counts.

---

## 12. Optimized kernel benchmark section

### Flow
Same pattern as the baseline:
1. synchronize
2. start timer
3. repeat launch:
   - `entropy_opt<16, 16><<<grids, blocks>>>(...)`
4. synchronize
5. stop timer
6. print average time

### Printed output
- `Average kernel (optimized) execution time ... (s)`

### Purpose
Measures the performance of the optimized implementation and allows direct comparison with the baseline.

---

## 13. Copying results back to host

After the optimized kernel completes:
- `cudaMemcpy(output, d_output, output_bytes, cudaMemcpyDeviceToHost)`

### Purpose
Retrieves the GPU-computed entropy map for validation.

Note that the copied output is from the **optimized kernel**, since it overwrites `d_output` after the baseline phase.

---

## 14. Device cleanup

The program frees:
- `d_input`
- `d_output`
- `d_logTable`

### Purpose
Releases GPU memory after computation is complete.

---

## 15. CPU reference verification

The program calls:
- `reference(output_ref, input, height, width);`

### Purpose
Computes a trusted CPU version of the entropy map for correctness checking.

The exact implementation is external, but based on use it should compute the same 5x5 local entropy.

---

## 16. Output comparison

The program compares each GPU output value against the CPU reference:
- tolerance: `1e-3f`

If any element differs by more than that:
- sets `ok = false`
- breaks out of the loops

Then prints:
- `PASS` if all match
- `FAIL` otherwise

### Purpose
Ensures the optimized GPU result is functionally correct.

---

## 17. Host cleanup and exit

The host arrays are freed:
- `input`
- `output`
- `output_ref`

Then the program returns `0`.

---

## Major data structures

## 1. Input image array
- Type: `char*`
- Host: `input`
- Device: `d_input`

### Role
Stores the discrete-valued 2D data over which local entropy is computed.

### Value range
Each element is in `[0, 15]`, representing one of 16 categories.

---

## 2. Output entropy array
- Type: `float*`
- Host: `output`
- Device: `d_output`

### Role
Stores one entropy value per input element.

---

## 3. Reference output array
- Type: `float*`
- Name: `output_ref`

### Role
Stores the CPU-computed correct answer for validation.

---

## 4. Histogram arrays in kernels

### Baseline kernel
- `char count[16]`

### Optimized kernel
- `__shared__ int sd_count[16][bsize_y*bsize_x]`

### Role
Track how many times each symbol value appears in the local 5x5 neighborhood.

---

## 5. Log lookup table
- Host: `float logTable[26]`
- Device: `float* d_logTable`

### Role
Stores `i * log2(i)` for counts `0..25`, with `0` for `i = 0, 1`.

This is used to speed up entropy calculation in the optimized kernel.

---

## Mathematical behavior

For each pixel, the program computes entropy of the local distribution:
- `H = -sum_k p_k log2(p_k)`

where:
- `p_k = count_k / total`
- `count_k` is how many times value `k` appears in the valid neighborhood
- `total` is number of valid cells in the local 5x5 window

At image borders, fewer than 25 cells contribute.

---

## Boundary handling

Both kernels handle edges by checking:
- `xx >= 0`
- `yy >= 0`
- `yy < height`
- `xx < width`

Only valid neighbors are counted.

This means:
- interior pixels use a full 25-cell window
- edge and corner pixels use smaller windows

---

## Relationship between baseline and optimized kernels

Both kernels are intended to produce the same entropy output.

### Baseline
- direct formula using probabilities
- simpler, more literal implementation

### Optimized
- algebraically transformed formula
- uses lookup table
- intended to reduce arithmetic overhead

The correctness test effectively validates the optimized kernel against the CPU reference.

---

## Important observations

## 1. The benchmark times both kernels, but verifies only the final output
Since `d_output` is overwritten by the optimized kernel, the final comparison checks the optimized kernel result, not the baseline result.

## 2. The optimized kernel still uses one thread per output pixel
There is no algorithmic change in output mapping, only in how entropy is computed internally.

## 3. Shared memory is used for per-thread histograms
This is not neighborhood tiling; instead, it gives each thread shared-memory-backed counters.

## 4. Input assumptions are strict
The kernel assumes all input values are valid histogram indices in `[0, 15]`. Since input is generated that way, this holds.

## 5. Maximum neighborhood size is 25
That explains:
- `char total`
- lookup table size 26
- safety of histogram count ranges

---

## Summary of major functions

### `entropy(...)`
Baseline CUDA kernel that computes local 5x5 entropy per element using direct histogram-to-probability calculation.

### `entropy_opt<16,16>(...)`
Templated optimized CUDA kernel that computes the same entropy using a lookup table and an equivalent formula based on histogram counts.

### `main(...)`
Parses arguments, builds random input, allocates host/device memory, runs both kernels repeatedly for timing, copies back results, validates against CPU reference, prints performance and pass/fail status, and frees resources.

### `reference(...)`
External CPU implementation used for correctness verification.

---

## Concise high-level summary

This is a CUDA entropy benchmark for a 2D discrete-valued image. It computes, for every pixel, the entropy of the values in its surrounding 5x5 neighborhood. The code includes:
- a straightforward baseline GPU kernel,
- a mathematically optimized GPU kernel using a precomputed log table,
- host-side timing for both,
- and CPU-based correctness verification.

Its main purpose is to compare performance of the two GPU implementations while ensuring the optimized version matches the expected result.


----- STARTING CODE INSERT - Invariant Assertions -----


=====> VERIFY THE PROMPTING <===== [tokens: 19,957]

TEMPERATURE: 0.2

SYSTEM: You are an expert C++/CUDA engineer performing surgical code insertion. Your ONLY task is to INSERT new instrumentation code at specified locations in an existing source file. You MUST preserve ALL existing code EXACTLY as written — do not modify, simplify, reorder, optimize, or remove any existing code, comments, blank lines, or whitespace. Every single line of the input file must appear in your output, in the same order. You always surround your generated code with the three characters ```.

PROMPT: 
    You are a GPU code verification specialist.

    Your task is to instrument the provided CUDA C++ source code with **transformation-invariant
    assertion checks** — properties that must hold in the original code AND any correct refactored
    version, regardless of parallelization strategy, memory layout, or algorithmic optimization.

    These a

```cpp
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <chrono>
#include <cuda.h>
#include "/home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/entropy/reference.h"

__global__ 
void entropy(
        float *__restrict__ d_entropy,
    const char*__restrict__ d_val, 
    int height, int width)
{
  const int x = threadIdx.x + blockIdx.x * blockDim.x;
  const int y = threadIdx.y + blockIdx.y * blockDim.y;

  // value of matrix element ranges from 0 inclusive to 16 exclusive
  char count[16];
  for (int i = 0; i < 16; i++) count[i] = 0;

  // total number of valid elements
  char total = 0;

  // 5x5 window
  for(int dy = -2; dy <= 2; dy++) {
    for(int dx = -2; dx <= 2; dx++) {
      int xx = x + dx;
      int yy = y + dy;
      if(xx >= 0 && yy >= 0 && yy < height && xx < width) {
        count[d_val[yy * width + xx]]++;
        total++;
      }
    }
  }

  float entropy = 0;
  if (total < 1) {
    total = 1;
  } else {
    for(int k = 0; k < 16; k++) {
      float p = __fdividef((float)count[k], (float)total);
      entropy -= p * log2f(p);
    }
  }

  if(y < height && x < width) d_entropy[y * width + x] = entropy;
}

template<int bsize_x, int bsize_y>
__global__ void entropy_opt(
       float *__restrict__ d_entropy,
  const  char*__restrict__ d_val, 
  const float*__restrict__ d_logTable,
  int m, int n)
{
  __shared__ int sd_count[16][bsize_y*bsize_x];

  const int x = threadIdx.x + blockIdx.x * blockDim.x;
  const int y = threadIdx.y + blockIdx.y * blockDim.y;
  const int idx = threadIdx.y*bsize_x + threadIdx.x;

  for(int i = 0; i < 16;i++) sd_count[i][idx] = 0;

  char total = 0;
  for(int dy = -2; dy <= 2; dy++) {
    for(int dx = -2; dx <= 2; dx++) {
      int xx = x + dx,
          yy = y + dy;

      if(xx >= 0 && yy >= 0 && yy < m && xx < n) {
        sd_count[d_val[yy*n+xx]][idx]++;
        total++;
      }
    }
  }

  float entropy = 0;
  for(int k = 0; k < 16; k++)
    entropy -= d_logTable[sd_count[k][idx]];
  
  entropy = entropy / total + log2f(total);
  if(y < m && x < n) d_entropy[y*n+x] = entropy;
}

int main(int argc, char* argv[]) {
  if (argc != 4) {
    printf("Usage: %s <width> <height> <repeat>\n", argv[0]);
    return 1;
  }
  const int width = atoi(argv[1]); 
  const int height = atoi(argv[2]); 
  const int repeat = atoi(argv[3]); 

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate parsed dimensions and repeat count at function entry
  {
      bool _t1_pass = (width > 0 && height > 0 && repeat > 0);
      printf("LASSI-VERIFY ASSERTIONS [main_dims_repeat_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  const int input_bytes = width * height * sizeof(char);
  const int output_bytes = width * height * sizeof(float);
  char* input = (char*) malloc (input_bytes);
  float* output = (float*) malloc (output_bytes);
  float* output_ref = (float*) malloc (output_bytes);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate host allocations and byte counts
  {
      bool _t1_pass = (input_bytes > 0 && output_bytes > 0 && input != NULL && output != NULL && output_ref != NULL);
      printf("LASSI-VERIFY ASSERTIONS [main_host_alloc_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  srand(123);
  for (int i = 0; i < height; i++)
    for (int j = 0; j < width; j++)
      input[i * width + j] = rand() % 16;

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate input symbol range before host-to-device transfer
  {
      bool _t1_pass = true;
      for (int i = 0; i < height * width; i++) {
          if (!(input[i] >= 0 && input[i] <= 15)) { _t1_pass = false; break; }
      }
      printf("LASSI-VERIFY ASSERTIONS [main_input_symbol_range_pre_htod]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  char* d_input;
  cudaMalloc((void**)&d_input, input_bytes);
  cudaMemcpy(d_input, input, input_bytes, cudaMemcpyHostToDevice);

  float* d_output;
  cudaMalloc((void**)&d_output, output_bytes);

  dim3 grids ((width+15)/16, (height+15)/16);
  dim3 blocks (16, 16);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate kernel launch configuration
  {
      bool _t1_pass = (grids.x > 0 && grids.y > 0 && blocks.x > 0 && blocks.y > 0 && blocks.x <= 1024 && blocks.y <= 1024);
      printf("LASSI-VERIFY ASSERTIONS [main_kernel_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // baseline kernel
  cudaDeviceSynchronize();
  auto start = std::chrono::steady_clock::now();
  
  for (int i = 0; i < repeat; i++)
    entropy <<< grids, blocks >>> (d_output, d_input, height, width);

  cudaDeviceSynchronize();
  auto end = std::chrono::steady_clock::now();
  auto time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  printf("Average kernel (baseline) execution time %f (s)\n", (time * 1e-9f) / repeat);

  // optimized kernel

  float logTable[26];
  for (int i = 0; i <= 25; i++) logTable[i] = i <= 1 ? 0 : i*log2f(i);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate log lookup table before host-to-device transfer
  {
      bool _t1_pass = true;
      if (!(logTable[0] == 0.0f && logTable[1] == 0.0f)) _t1_pass = false;
      for (int i = 0; i <= 25 && _t1_pass; i++) {
          if (!isfinite(logTable[i])) { _t1_pass = false; break; }
      }
      printf("LASSI-VERIFY ASSERTIONS [main_logtable_valid_pre_htod]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  float* d_logTable;
  cudaMalloc((void**)&d_logTable, sizeof(logTable));
  cudaMemcpy(d_logTable, logTable, sizeof(logTable), cudaMemcpyHostToDevice);
 
  cudaDeviceSynchronize();
  start = std::chrono::steady_clock::now();

  for (int i = 0; i < repeat; i++)
    entropy_opt<16, 16> <<< grids, blocks >>> (d_output, d_input, d_logTable, height, width);

  cudaDeviceSynchronize();
  end = std::chrono::steady_clock::now();
  time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  printf("Average kernel (optimized) execution time %f (s)\n", (time * 1e-9f) / repeat);

  cudaMemcpy(output, d_output, output_bytes, cudaMemcpyDeviceToHost);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate output values after device-to-host transfer are finite and within entropy bounds
  {
      bool _t1_pass = true;
      const int _t1_total_elems = height * width;
      const float _t1_max_entropy = log2f(16.0f);
      for (int i = 0; i < _t1_total_elems; i++) {
          if (!isfinite(output[i]) || output[i] < -1e-4f || output[i] > _t1_max_entropy + 1e-3f) { _t1_pass = false; break; }
      }
      printf("LASSI-VERIFY ASSERTIONS [main_output_entropy_bounds_post_dtoh]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaFree(d_input);
  cudaFree(d_output);
  cudaFree(d_logTable);

  // verify
  reference(output_ref, input, height, width);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate CPU reference output is finite and within entropy bounds
  {
      bool _t1_pass = true;
      const int _t1_total_elems = height * width;
      const float _t1_max_entropy = log2f(16.0f);
      for (int i = 0; i < _t1_total_elems; i++) {
          if (!isfinite(output_ref[i]) || output_ref[i] < -1e-4f || output_ref[i] > _t1_max_entropy + 1e-3f) { _t1_pass = false; break; }
      }
      printf("LASSI-VERIFY ASSERTIONS [main_reference_entropy_bounds_post_compute]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  bool ok = true;
  for (int i = 0; i < height; i++) {
    for (int j = 0; j < width; j++) {
      if (fabsf(output[i * width + j] - output_ref[i * width + j]) > 1e-3f) {
        ok = false; 
        break;
      }
    }
    if (!ok) break;
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - Validate final GPU and reference outputs agree within stated tolerance
  {
      bool _t1_pass = ok;
      printf("LASSI-VERIFY ASSERTIONS [main_final_reference_match]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  printf("%s\n", ok ? "PASS" : "FAIL");
 
  free(input);
  free(output);
  free(output_ref);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (VERIFICATION BLOCK — do not edit or delete this block; this block and its assertion labels must remain exactly as written) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
      printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<
  return 0;
}
```


----- STARTING CODE INSERT - Passthrough Data Dump Blocks -----


=====> VERIFY THE PROMPTING <===== [tokens: 14,572]

TEMPERATURE: 0.2

SYSTEM: You are an expert C++/CUDA engineer performing surgical code insertion. Your ONLY task is to INSERT new instrumentation code at specified locations in an existing source file. You MUST preserve ALL existing code EXACTLY as written — do not modify, simplify, reorder, optimize, or remove any existing code, comments, blank lines, or whitespace. Every single line of the input file must appear in your output, in the same order. You always surround your generated code with the three characters ```.

PROMPT: 
        You are preparing GPU-accelerated scientific code for Tier 2 empirical equivalence checking
        as part of the LASSI-Verify verification framework.

        Your task is to instrument the source code with **passthrough data blocks** at key observation
        points. These blocks capture computational state to stdout in a structured